In [1]:
from matplotlib.backends.backend_pdf import PdfPages
import argparse
import pathlib
import importlib

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import json
import uproot
import sys
sys.path.append("../../scripts/")

from pathlib import Path
import common_functions as au
from baseline_chi2pid import passes_kplus_chi2pid_cut

In [2]:
cols = ["pid", "p", "theta", "beta", "chi2pid", "rich_RQ", "vz", "bdt_pass", "rich_best_PID", "rich_RQ", "rich_best_ntot"]
kinematics =["Mx_eKX","Mx_epiX","Mx_epX", "Q2", "W", "y"]

for kin in kinematics:
    cols.append(kin)
    
df = uproot.open("~/ML_Files/epkx_data/scored/epkx_dataset.root:PhysicsEvents").arrays(cols, library="pd")
df=df[df["rich_best_ntot"]>2]
df=df[df["rich_RQ"]>0.1]
df=au.apply_Sidis_Cuts(df)

outDir="../../figures/Data_Application/"

In [3]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Define momentum bins
pEdges = au.makeBinEdges(3, 5, 10)

# Select RICH angular range
df_rich = df[df["theta"] < 20]
df_rich = df_rich[
    (df_rich["rich_best_PID"] == 321) |
    (df_rich["rich_best_PID"] == 221) |
    (df_rich["rich_best_PID"] == 2212)
]



vals=[]
errs=[]
pCenters = []

# Split into bins
pBins = au.makeBins(df_rich, "p", binEdges=pEdges)

for i, pbin in enumerate(pBins):

    numerator_df = pbin[
        (pbin["pid"] == 321) &
        (pbin["bdt_pass"] == True) &
        (pbin["rich_best_PID"] != 321)
    ]

    denominator_df = pbin[
        (pbin["pid"] == 321) &
        (pbin["bdt_pass"] == True)
    ]

    numerator = len(numerator_df)
    denominator = len(denominator_df)

    r = 0
    rErr = 0

    if denominator != 0:
        r = numerator / denominator

        if numerator != 0:
            rErr = r * math.sqrt((1/numerator) + (1/denominator))

    vals.append(r)
    errs.append(rErr)

    pCenters.append((pEdges[i] + pEdges[i+1]) / 2)


# Plot
plt.figure(figsize=(7,5))

plt.errorbar(
    pCenters,
    vals,
    yerr=errs,
    fmt='o',
    capsize=4,
    markersize=6
)

plt.xlabel("Momentum p (GeV)")
plt.ylabel("Kaon contamination")
plt.title(r"BDT RICH Truth contamination theta < 20")

# Put ticks at bin edges
plt.xticks(pEdges)

# Optional: make sure plot spans exactly the bin range
plt.xlim(pEdges[0], pEdges[-1])

plt.grid(False)

# Save figure
plt.savefig(outDir+"bdt_rich_contamination.png",
            dpi=150,
            bbox_inches="tight")

plt.show()

FieldNotFoundError: no field 'mc_matching_pid' in record with 16 fields

In [ ]:
import uproot
import matplotlib.pyplot as plt

# Columns needed
cols = [
    "pid",
    "p",
    "theta",
    "beta",
    "chi2pid",
    "rich_RQ",
    "vz",
    "bdt_pass",
    "rich_best_PID",
    "Mx_eKX",
    "Mx_epiX",
    "Mx_epX",
    "Q2",
    "W",
    "y",
    "rich_best_ntot",
    "rich_RQ"
]

# Load scored pion sample
df = uproot.open(
    "~/ML_Files/data_epiN_v01/scored/epiN_dataset.root:PhysicsEvents"
).arrays(cols, library="pd")
df = df[df["rich_best_ntot"]>2.5]
df = df[df["rich_RQ"]>0.1]
outDir = "../../figures/Data_Application/"


# Apply momentum and truth-pion selection
pCut_all = df[
    (df["p"] > 3)&
    (df["theta"]<20)&
    (df["pid"] == 211)]

# True pions that the BDT accepts as kaons
pCut_bdt = df[
    (df["p"] > 3) &
    (df["theta"]<20)&
    (df["bdt_pass"] == True)&
    (df["pid"] == 211)
]

pCut_rich = df[
    (df["p"] > 3) &
    (df["theta"] < 20) &
    (df["pid"] == 211) &
    (df["rich_best_PID"] == 321)
]



# Plot neutron missing-mass peak
plt.figure(figsize=(7,5))

plt.hist(
    pCut_all["Mx_epiX"],
    bins=100,
    histtype="step",
    label=r"All $\pi^+$ (RICH acceptance Cuts)",
    density=False
)

plt.hist(
    pCut_bdt["Mx_epiX"],
    bins=100,
    histtype="step",
    label=r"$\pi^+$ passing K BDT",
    density=False
)

plt.hist(
    pCut_rich["Mx_epiX"],
    bins=100,
    histtype="step",
    label=r"$\pi^+$ with RICH $K^+$ PID",
    density=False
)


plt.xlabel(r"$M_x(e\pi^+X)$ [GeV]")
plt.ylabel("Normalized counts")
plt.title(r"Neutron missing mass: $ep\rightarrow e\pi^+(n)$ ($p>3$ GeV)")

plt.legend()
plt.grid(False)

plt.savefig(
    outDir + "Mx_epiN_BDT_misID.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()
plt.close()